# Week 1 - Local Inference with Ollama

- **Non-local (API) calls.** OpenAI, Anthropic, Google. Nothing to install beyond Python packages.
- **Local calls with Ollama.** Installing Ollama, running your first model, then calling it from Python.

By the end, you'll have classified the same dataset two ways (API + Ollama) and be able to compare them side by side.

## Learning objectives

1. Set up API credentials safely.
2. Write a unified function that classifies text using OpenAI, Anthropic, or Gemini.
3. Force models to return **structured** output instead of free text.
4. Handle **retries / rate limits** so a flaky network doesn't kill a long run.
5. Track **token usage and cost** before scaling up an API-based run.
6. Install Ollama and run your **first local model**, with zero prior setup.
7. Call a local Ollama model from Python using the **same pipeline shape** as the API calls.
8. Compare API vs. local models on **cost, privacy, and performance**.

## How this notebook is structured

Every model-calling function (API or Ollama) has a **MOCK mode** and a **LIVE mode**, controlled by one flag near the top. In MOCK mode nothing leaves your computer and no software needs to be installed a fake "model" simulates realistic responses so you can build and test the *entire* pipeline, including Ollama, before installing anything. Flip one flag to `False`, install the real software, and the same pipeline code runs for real.

**Run the whole notebook in MOCK mode first.** Then work through the installation steps for real, flip the flag, and re-run.


---
# Part A - Non-Local (API) Calls
---

## A0. The privacy tradeoff

Every API provider is a **third party**. When you call their API, the text you're classifying (e.g., a classroom transcript, an open-ended survey response, a student essay) leaves your machine and is processed on their servers.

Before running real data through any API, ask:

- Does your IRB protocol / data use agreement allow sending this data to a third party?
- Does the provider **train on your data** by default? (Usually off by default for paid API usage, but this can vary.)
- Could you de-identify the text first and still answer your research question?
- Is there a **zero data retention (ZDR)** or enterprise agreement available if your data is sensitive?


In [2]:
# ---------------------------------------------------------------------------
# Setup
# ---------------------------------------------------------------------------
# Run this once. In LIVE mode for Part A you'll also need:
#   pip install openai anthropic google-generativeai python-dotenv
# In LIVE mode for Part B you'll additionally need Ollama installed (Part B covers this).

import os
import time
import json
import random
import hashlib
import subprocess
import shutil

import pandas as pd
from pydantic import BaseModel, Field
from tenacity import retry, wait_random_exponential, stop_after_attempt, retry_if_exception_type

random.seed(42)

# THE ONE FLAG THAT MATTERS
USE_MOCK = True   # <- set to False only once you have real API keys AND/OR Ollama installed

print(f"Running in {'MOCK' if USE_MOCK else 'LIVE'} mode.")


Running in MOCK mode.


## A1. API keys

**Never** hardcode API keys in a notebook, and never commit them to git (including private repos -- assume they leak eventually).

1. Create a file named `.env` in this folder.
2. Add your keys:
   ```
   OPENAI_API_KEY=sk-...
   ANTHROPIC_API_KEY=sk-ant-...
   GOOGLE_API_KEY=AI...
   ```
3. Add `.env` to `.gitignore` **before** you ever `git add .`
4. Load it with `python-dotenv` (below).

If you're on shared/HPC systems: don't put real keys in a job script others can read. Use a restricted-permission file (`chmod 600 .env`).


In [ ]:
# In LIVE mode, uncomment:
#
# from dotenv import load_dotenv
# load_dotenv()
#
# api_keys = {
#     "openai": os.environ.get("OPENAI_API_KEY"),
#     "anthropic": os.environ.get("ANTHROPIC_API_KEY"),
#     "google": os.environ.get("GOOGLE_API_KEY"),
# }
# missing = [k for k, v in api_keys.items() if not v]
# if missing:
#     raise EnvironmentError(f"Missing API keys for: {missing}. Check your .env file.")

if USE_MOCK:
    print("MOCK mode: no API keys needed, no network calls will be made.")


## A2. The dataset

Same task throughout this notebook: rate a book review from 0 (bad) to 5 (good).


In [ ]:
reviews = [
    ("Absolutely loved this book. Could not put it down, finished it in two days.", 5),
    ("This was a decent read but the pacing dragged in the middle third.", 3),
    ("I could not get through this. The characters felt flat and the plot made no sense.", 1),
    ("A solid mystery novel with a satisfying twist at the end.", 4),
    ("Terrible. Would not recommend to anyone. Waste of time.", 0),
    ("Pretty good overall, some parts felt rushed but the ending redeemed it.", 3),
    ("One of the best books I've read this year. Beautifully written.", 5),
    ("It was fine. Nothing special, nothing terrible. Forgettable.", 2),
    ("The world-building was incredible but the dialogue felt stilted.", 3),
    ("I really disliked the main character's decisions throughout. Frustrating read.", 1),
    ("A masterpiece. Every chapter built on the last perfectly.", 5),
    ("Okay premise, poor execution. The ending felt rushed and unearned.", 2),
    ("Charming and funny, exactly what I needed. Would read again.", 4),
    ("Confusing structure made this hard to follow. Gave up halfway through.", 1),
    ("Solid entry in the series, though not as strong as the first book.", 3),
]

df = pd.DataFrame(reviews, columns=["review_text", "true_rating"])
df.head()


## A3. Define the Output Format

Decide your output schema **before** you write a prompt. If you don't, you'll get back a mix of `"I'd rate this a 4"`, `"Rating: 4/5"`, and `"4"` -- formats you now have to parse with regex, and regex on LLM output breaks constantly.


In [ ]:
class RatingClassification(BaseModel):
    rating: int = Field(..., ge = 0, le = 5, description = "Predicted rating from 0 (bad) to 5 (good)")
    confidence: str = Field(..., description = "One of: low, medium, high")
    rationale: str = Field(..., description = "One sentence justification")

# Every provider function -- OpenAI, Anthropic, Google, and (Part B) Ollama --
# returns one of these. That's the "unified interface" idea: your evaluation
# code never needs to know which provider or which deployment method produced it.

RatingClassification(rating = 4, confidence = "high", rationale = "Positive language throughout, minor complaints only.")


## A4. The prompt

Same prompt, same case, sent to every provider (API-based *and*, in Part B, local) so any performance difference is attributable to the model, not to inconsistent instructions. Week 3 (systematic prompt engineering) formalizes this idea further.


In [ ]:
SYSTEM_PROMPT = (
    "You are a careful annotator rating book reviews. "
    "Given a review, predict the rating on a 0 (bad) to 5 (good) scale, "
    "matching how the reviewer likely rated the book themselves. "
    "Respond only in the requested structured format."
)

def build_user_prompt(review_text: str) -> str:
    return f"Book review:\n\"\"\"\n{review_text}\n\"\"\"\n\nPredict the rating."


## A5. The mock backend

Simulates a model's structured response deterministically (seeded by the text and provider name) so you can build and debug the whole pipeline -- retries, rate limiting, cost tracking -- with zero network access and zero installed software. It occasionally raises a fake rate-limit error so the retry logic has something to do. This same mock function is reused for Ollama in Part B.


In [ ]:
class FakeRateLimitError(Exception):
    pass

def _mock_model_call(review_text: str, provider: str) -> dict:
    '''Simulates a model's structured response. Deterministic per (text, provider).'''
    seed = int(hashlib.sha256((review_text + provider).encode()).hexdigest(), 16)
    rng = random.Random(seed)
    if rng.random() < 0.15:
        raise FakeRateLimitError(f"[{provider}] 429 Too Many Requests (simulated)")

    positive_words = ["loved", "best", "incredible", "beautiful", "masterpiece", "solid", "charming", "good", "great"]
    negative_words = ["terrible", "waste", "disliked", "confusing", "flat", "frustrating", "poor", "hate"]
    text_lower = review_text.lower()
    score = 2.5 + sum(w in text_lower for w in positive_words) - sum(w in text_lower for w in negative_words)
    score = max(0, min(5, round(score + rng.uniform(-0.5, 0.5))))

    prompt_tokens = len(SYSTEM_PROMPT.split()) + len(review_text.split()) + 15
    completion_tokens = 20

    return {
        "rating": int(score),
        "confidence": rng.choice(["low", "medium", "high"]),
        "rationale": "Simulated rationale based on sentiment cues in the review.",
        "usage": {"prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens},
    }


## A6. Real provider calls (reference - used when `USE_MOCK = False`)

The actual API call patterns for each provider, as of early 2026. Notice the shape is the same across all three: system + user message in, structured object + token usage out. That symmetry is what lets `classify_review()` not care which provider it's talking to -- and it's the same symmetry Part B extends to Ollama.


In [ ]:
def _openai_call(review_text: str) -> dict:
    from openai import OpenAI
    client = OpenAI()  # reads OPENAI_API_KEY from environment

    completion = client.chat.completions.parse(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(review_text)},
        ],
        response_format=RatingClassification,
    )
    parsed = completion.choices[0].message.parsed
    usage = completion.usage
    return {
        **parsed.model_dump(),
        "usage": {"prompt_tokens": usage.prompt_tokens, "completion_tokens": usage.completion_tokens},
    }


def _anthropic_call(review_text: str) -> dict:
    import anthropic
    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from environment

    tool_schema = {
        "name": "record_rating",
        "description": "Record the predicted rating for a book review.",
        "input_schema": RatingClassification.model_json_schema(),
    }
    message = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=300,
        system=SYSTEM_PROMPT,
        tools=[tool_schema],
        tool_choice={"type": "tool", "name": "record_rating"},
        messages=[{"role": "user", "content": build_user_prompt(review_text)}],
    )
    tool_use_block = next(b for b in message.content if b.type == "tool_use")
    parsed = tool_use_block.input
    return {
        **parsed,
        "usage": {
            "prompt_tokens": message.usage.input_tokens,
            "completion_tokens": message.usage.output_tokens,
        },
    }


def _gemini_call(review_text: str) -> dict:
    import google.generativeai as genai
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
    model = genai.GenerativeModel(
        "gemini-2.5-flash",
        system_instruction=SYSTEM_PROMPT,
        generation_config={
            "response_mime_type": "application/json",
            "response_schema": RatingClassification.model_json_schema(),
        },
    )
    response = model.generate_content(build_user_prompt(review_text))
    parsed = json.loads(response.text)
    usage = response.usage_metadata
    return {
        **parsed,
        "usage": {
            "prompt_tokens": usage.prompt_token_count,
            "completion_tokens": usage.candidates_token_count,
        },
    }


## A7. The unified pipeline function

One function, `classify_review()`, that works the same way no matter which provider (or, after Part B, which deployment method) you point it at, and that survives transient errors instead of crashing your whole run partway through.


In [ ]:
@retry(
    retry = retry_if_exception_type(FakeRateLimitError),  # in LIVE mode: catch the real provider's RateLimitError classes
    wait = wait_random_exponential(min=1, max=20),
    stop = stop_after_attempt(5),
    reraise = True,
)

def classify_review(review_text: str, provider: str = "openai") -> dict:
    '''
    Classify a single book review using the given provider.
    Returns a dict matching RatingClassification's fields, plus a 'usage' dict.
    Retries automatically on rate-limit errors.
    provider: one of "openai", "anthropic", "google", "ollama"
    '''
    if USE_MOCK:
        return _mock_model_call(review_text, provider)

    if provider == "openai":
        return _openai_call(review_text)
    elif provider == "anthropic":
        return _anthropic_call(review_text)
    elif provider == "google":
        return _gemini_call(review_text)
    elif provider == "ollama":
        return _ollama_call(review_text)   # defined in Part B
    else:
        raise ValueError(f"Unknown provider: {provider}")


In [ ]:
# check on one review, one provider
result = classify_review(df.loc[0, "review_text"], provider="openai")
result


## A8. Running the pipeline over a dataset

Two things a single test call doesn't show you:

- **Rate limiting between calls.** A small `time.sleep()` between calls, even with retries, avoids tripping rate limits in the first place.
- **Graceful failure per-row.** If row 200 of 500 fails even after retries, you want a `None`/error marker in that row, not a crashed script that loses your first 199 results.


In [ ]:
def run_pipeline(dataframe: pd.DataFrame, provider: str, sleep_seconds: float = 0.0) -> pd.DataFrame:
    ratings, confidences, rationales, errors = [], [], [], []
    total_prompt_tokens = 0
    total_completion_tokens = 0

    for i, review_text in enumerate(dataframe["review_text"]):
        try:
            result = classify_review(review_text, provider=provider)
            ratings.append(result["rating"])
            confidences.append(result["confidence"])
            rationales.append(result["rationale"])
            total_prompt_tokens += result["usage"]["prompt_tokens"]
            total_completion_tokens += result["usage"]["completion_tokens"]
            errors.append(None)
        except Exception as e:
            ratings.append(None)
            confidences.append(None)
            rationales.append(None)
            errors.append(str(e))
        if sleep_seconds:
            time.sleep(sleep_seconds)

    out = dataframe.copy()
    out[f"{provider}_rating"] = ratings
    out[f"{provider}_confidence"] = confidences
    out[f"{provider}_rationale"] = rationales
    out[f"{provider}_error"] = errors

    print(f"[{provider}] prompt tokens: {total_prompt_tokens}, completion tokens: {total_completion_tokens}")
    return out, total_prompt_tokens, total_completion_tokens

results_openai, pt_openai, ct_openai = run_pipeline(df, provider="openai")
results_openai[["review_text", "true_rating", "openai_rating", "openai_confidence", "openai_error"]]


## A9. Cost & token accounting

Before pointing a pipeline like this at 5,000 transcripts, know what it will cost. **Prices change often** -- check each provider's current pricing page rather than trusting a remembered number. The pattern below is what matters; update the numbers before a real run.


In [ ]:
# example prices in USD per 1M tokens -- ILLUSTRATIVE ONLY. Check current pricing before relying on this.
PRICING_PER_MILLION_TOKENS = {
    "openai":    {"input": 0.40, "output": 1.60},
    "anthropic": {"input": 3.00, "output": 15.00},
    "google":    {"input": 0.30, "output": 2.50},
    "ollama":    {"input": 0.00, "output": 0.00},  # no per-token API cost -- see Part B discussion on hardware cost instead
}

def estimate_cost(provider: str, prompt_tokens: int, completion_tokens: int) -> float:
    rates = PRICING_PER_MILLION_TOKENS[provider]
    return (prompt_tokens / 1_000_000) * rates["input"] + (completion_tokens / 1_000_000) * rates["output"]

cost_this_run = estimate_cost("openai", pt_openai, ct_openai)
n_rows = len(df)
print(f"Cost for this {n_rows}-row test run: ${cost_this_run:.6f}")
print(f"Extrapolated cost for 5,000 rows: ${cost_this_run / n_rows * 5000:.2f}")


## A10. Running the other two API providers

Repeat the run for `anthropic` and `google`. In MOCK mode this demonstrates the workflow; in LIVE mode this is a real three-way comparison.


In [ ]:
results_anthropic, pt_a, ct_a = run_pipeline(df, provider="anthropic")
results_google, pt_g, ct_g = run_pipeline(df, provider="google")

comparison = df.copy()
comparison["openai_rating"] = results_openai["openai_rating"]
comparison["anthropic_rating"] = results_anthropic["anthropic_rating"]
comparison["google_rating"] = results_google["google_rating"]
comparison


## A11. An intro to performance evaluation

Just enough evaluation to know whether the pipeline is working: exact-match accuracy, mean absolute error, and RMSE.


In [ ]:
import math

def evaluate(dataframe: pd.DataFrame, pred_col: str, true_col: str = "true_rating") -> dict:
    valid = dataframe.dropna(subset=[pred_col])
    n = len(valid)
    if n == 0:
        return {"n": 0, "accuracy": None, "mae": None, "rmse": None}
    errors = valid[pred_col] - valid[true_col]
    accuracy = (errors == 0).mean()
    mae = errors.abs().mean()
    rmse = math.sqrt((errors ** 2).mean())
    return {"n": n, "accuracy": round(accuracy, 3), "mae": round(mae, 3), "rmse": round(rmse, 3)}

for provider_col in ["openai_rating", "anthropic_rating", "google_rating"]:
    print(provider_col, evaluate(comparison, provider_col))


---
# Part B - Introduction to Local Inference with Ollama
---

Everything in Part A ran on someone else's servers. Part B runs a model **on your own machine**, with no API key and no per-call cost. This section assumes nothing beyond what you just built in Part A.

## B0. What's actually different about a local model?

| | API (Part A) | Local / Ollama (Part B) |
|---|---|---|
| Where does the text go? | Sent to a third-party server | Stays on your machine |
| Per-call cost? | Yes, billed per token | No cost (you already own/rent the hardware) |
| Setup required? | Just an API key | Install software, download a multi-GB model file |
| Model quality/size available? | State-of-the-art, provider's choice | Limited by *your* hardware |
| Reproducibility? | Provider can silently update the model | You control exactly which model/version runs |


## B1. Installing Ollama

Ollama is a program that downloads open-weight models and runs them locally, exposing a simple API (and command-line tool) so you don't have to hand-write model-loading code yourself.

**This step happens on your own computer, outside this notebook**. There's no code cell for "install Ollama," because it's an installer/download, not a Python package.

- **macOS:** download the app from [ollama.com](https://ollama.com), open it, follow the installer.
- **Windows:** download the installer from [ollama.com](https://ollama.com) and run it.
- **Linux:** `curl -fsSL https://ollama.com/install.sh | sh`

After installing, verify it worked by opening a terminal and running:

```bash
ollama --version
```

If that prints a version number, you're set. If it says "command not found," the install didn't complete or your terminal needs to be restarted.


In [ ]:
# check for Ollama's presence from Python -- this cell is safe to run either way.
# in MOCK mode it's just informational; it doesn't block anything below.

ollama_installed = shutil.which("ollama") is not None
print(f"Ollama executable found on PATH: {ollama_installed}")
if not ollama_installed and not USE_MOCK:
    print("Install Ollama first (see markdown above), then re-run this cell.")


Ollama executable found on PATH: True


## B2. Your first model, from the command line

Before touching Python, get comfortable with the `ollama` CLI directly.

In a terminal:

```bash
# Download a small model (~2GB). "3B" means 3 billion parameters
# what that means for hardware requirements. This one runs on most modern laptops.
ollama pull llama3.2

# See what models you've downloaded
ollama list

# Chat with it interactively
ollama run llama3.2
```

Try asking it something simple, like "Hi, how are you?" -- this is the same sanity check `test_ollama.py` in your existing repo runs on HPC. Type `/bye` to exit the chat.

**What just happened:** 
- `ollama pull` downloaded a quantized model file (we will talk more about quantization later).
- `ollama run` started a local server in the background *and* opened an interactive chat with it. That background server is what Python will talk to next.


## B3. Calling Ollama from Python

Ollama exposes a REST API on `localhost:11434` (this is why it needs to be running -- either via `ollama run` or `ollama serve` -- before Python can reach it). There are two ways to talk to it:

1. **Raw HTTP requests** -- useful for understanding what's actually happening.
2. **The `ollama` Python package** -- a thin wrapper around the same API. `pip install ollama`.

Below is the raw-HTTP version first (so you can see there's no magic -- it's just a local web server), then the version we'll actually use in the pipeline.


In [ ]:
# Reference only -- shows what the ollama package does under the hood.
# (Not executed here; requires Ollama actually running.)

RAW_HTTP_EXAMPLE = '''
import requests

response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "llama3.2",
        "messages": [{"role": "user", "content": "Hi, how are you?"}],
        "stream": False,
    },
)
print(response.json()["message"]["content"])
'''
print(RAW_HTTP_EXAMPLE)


## B4. Structured output from Ollama

Just like Part A, we want a `RatingClassification` object back, not free text to regex-parse. Ollama supports constraining output to a JSON schema via the `format` parameter (available in recent Ollama versions -- verify against current docs, since this is exactly the kind of feature that improves quickly).


In [ ]:
def _ollama_call(review_text: str, model: str = "llama3.2") -> dict:
    import ollama  # pip install ollama

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(review_text)},
        ],
        format=RatingClassification.model_json_schema(),  # constrains output to match the schema
    )
    parsed = RatingClassification.model_validate_json(response["message"]["content"])
    usage = response.get("prompt_eval_count", 0), response.get("eval_count", 0)
    return {
        **parsed.model_dump(),
        "usage": {"prompt_tokens": usage[0], "completion_tokens": usage[1]},
    }


Notice this function has the exact same shape as `_openai_call`, `_anthropic_call`, and `_gemini_call` from Part A: text in, `RatingClassification`-shaped dict out, plus a usage dict. That's why `classify_review()` in Section A7 already had an `"ollama"` branch waiting for it -- the *pipeline itself never needed to change* to add a fourth, completely different deployment method. This is the payoff of designing the schema and interface first, back in A3.


In [ ]:
# Run the same pipeline, same dataset, now against Ollama.
# In MOCK mode this reuses the same simulated backend as Part A (seeded on "ollama" as the provider name).
# In LIVE mode, make sure `ollama run llama3.2` (or `ollama serve`) is running in a terminal first.

results_ollama, pt_o, ct_o = run_pipeline(df, provider="ollama")
results_ollama[["review_text", "true_rating", "ollama_rating", "ollama_confidence", "ollama_error"]]


## B5. Putting it all together: four methods, one table

This is the comparison the workshop's Activities 1-3 build toward, now with real (or mock) numbers of your own.


In [ ]:
full_comparison = comparison.copy()
full_comparison["ollama_rating"] = results_ollama["ollama_rating"]

for provider_col in ["openai_rating", "anthropic_rating", "google_rating", "ollama_rating"]:
    metrics = evaluate(full_comparison, provider_col)
    print(f"{provider_col:20s} {metrics}")

full_comparison


## B6. Troubleshooting common first-time Ollama issues

- **"connection refused" from Python:** Ollama isn't running. Start it with `ollama serve` (or leave a `ollama run <model>` session open) in a terminal.
- **Model download seems stuck:** large models (7B+) can take a while on slow connections; check `ollama list` in another terminal to see if it eventually appears.
- **`ollama: command not found` after installing:** restart your terminal, or on Linux/macOS check that the install script added Ollama to your `PATH`.
- **Out of memory / very slow responses:** your hardware may not comfortably fit the model you chose. Week 6 covers how to estimate this *before* you pull a model, so you're not guessing.


## 📓 Week 1 Assignment

- Find a dataset you want to use. 
- Describe the classification task, answering the following questions:
    - Pretend you need to train human coders to classify the data. Write out the task to include any information the human coders may about each category to accurately classify the data.
        - What information (i.e., variable(s)) will the coders use to make classifications?
        - How should classification be formatted?
    - Which variable contains the correct classifications?
    
   